# QLoRA Fine-Tuning for Sepsis Prediction

This notebook fine-tunes Meditron-7B using QLoRA (4-bit quantization) for multi-label sepsis prediction.

## What This Notebook Does:
1.  Loads Meditron-7B with 4-bit quantization (saves memory)
2.  Adds LoRA adapters for efficient fine-tuning
3.  Trains on sepsis prediction data (h6, h12, h24, h48)
4.  Evaluates on validation set
5.  Saves the trained model

## Requirements:
- GPU: A10 (24GB) or similar
- Data: train.jsonl, val.jsonl in s3_bucket/data/

---
## Step 1: Install Required Packages

Run this cell first to install all dependencies.

In [1]:
# Install required packages
!pip install -q torch transformers accelerate peft bitsandbytes sentencepiece tqdm scikit-learn tensorboard boto3 s3fs

## Step 2: Import Libraries

In [1]:
import torch
import torch.nn as nn
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime
import boto3
import tempfile
import os

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Test S3 access
try:
    s3 = boto3.client('s3')
    print("\n✅ S3 client initialized successfully!")
    print("Testing S3 access...")
    response = s3.list_objects_v2(Bucket='mimic-data-nahiyan', Prefix='data/', MaxKeys=5)
    print(f"✅ S3 access working! Found {response.get('KeyCount', 0)} objects")
except Exception as e:
    print(f"❌ S3 access error: {e}")

✅ Libraries imported successfully!
PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA A10G
GPU Memory: 23.68 GB

✅ S3 client initialized successfully!
Testing S3 access...
✅ S3 access working! Found 5 objects


---
## Step 3: Configuration

In [2]:
# ========== CONFIGURATION ==========

# S3 Paths
S3_BUCKET = "mimic-data-nahiyan"
S3_MODEL_PREFIX = "models/meditron-7b"
S3_DATA_PREFIX = "data"

# Data files in S3
TRAIN_FILE = "sepsis_top_n_train.jsonl"
VAL_FILE = "sepsis_top_n_val.jsonl"
TEST_FILE = "sepsis_top_n_test.jsonl"

# Local paths for caching/output
LOCAL_CACHE_DIR = "/home/ubuntu/nahiyan_workspace/sepsis_prediction/model_cache"
LOCAL_DATA_DIR = "/home/ubuntu/nahiyan_workspace/sepsis_prediction/data_cache"
OUTPUT_DIR = "/home/ubuntu/nahiyan_workspace/sepsis_prediction/outputs/qlora_s3"

# Training hyperparameters (OPTIMIZED with class imbalance fixes)
NUM_EPOCHS = 3  # Changed from 1 to 3 for better convergence
BATCH_SIZE = 2  # Per device train batch size (safe for 24GB GPU)
EVAL_BATCH_SIZE = 2  # Per device eval batch size
GRADIENT_ACCUMULATION_STEPS = 32  # Effective batch size = 2 * 32 = 64
LEARNING_RATE = 5e-5  # Reduced from 2e-4 for stability with oversampling
MAX_LENGTH = 1024  # Reduced from 2048 for speed

# QLoRA config
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

# Evaluation
EVALUATION_STRATEGY = "epoch"  # Evaluate once per epoch
SAVE_STRATEGY = "epoch"  # Save at end of epoch
LOGGING_STEPS = 50

# Data sampling for faster training
USE_DATA_SAMPLING = False  # Disabled - using oversampling instead
SAMPLE_RATIO = 1.0  # Use 100% of data

# Seed for reproducibility
SEED = 42

# Create directories
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Configuration set!")
print(f"\nS3 Configuration:")
print(f"  Bucket: s3://{S3_BUCKET}")
print(f"  Model: s3://{S3_BUCKET}/{S3_MODEL_PREFIX}")
print(f"  Train data: s3://{S3_BUCKET}/{S3_DATA_PREFIX}/{TRAIN_FILE}")
print(f"  Val data: s3://{S3_BUCKET}/{S3_DATA_PREFIX}/{VAL_FILE}")
print(f"  Test data: s3://{S3_BUCKET}/{S3_DATA_PREFIX}/{TEST_FILE}")
print(f"\nLocal Configuration:")
print(f"  Model cache: {LOCAL_CACHE_DIR}")
print(f"  Data cache: {LOCAL_DATA_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"\nTraining Configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Data sampling: {USE_DATA_SAMPLING} (using oversampling instead)")
print(f"  Evaluation: {EVALUATION_STRATEGY}")
print(f"\n⏱️  Expected training time: ~10-12 hours (with oversampled data)")
print(f"💡 Class imbalance fixes applied: weighted loss + oversampling + lower thresholds")

✅ Configuration set!

S3 Configuration:
  Bucket: s3://mimic-data-nahiyan
  Model: s3://mimic-data-nahiyan/models/meditron-7b
  Train data: s3://mimic-data-nahiyan/data/sepsis_top_n_train.jsonl
  Val data: s3://mimic-data-nahiyan/data/sepsis_top_n_val.jsonl
  Test data: s3://mimic-data-nahiyan/data/sepsis_top_n_test.jsonl

Local Configuration:
  Model cache: /home/ubuntu/nahiyan_workspace/sepsis_prediction/model_cache
  Data cache: /home/ubuntu/nahiyan_workspace/sepsis_prediction/data_cache
  Output: /home/ubuntu/nahiyan_workspace/sepsis_prediction/outputs/qlora_s3

Training Configuration:
  Epochs: 3
  Batch size: 2
  Effective batch size: 64
  Max length: 1024
  Learning rate: 5e-05
  Data sampling: False (using oversampling instead)
  Evaluation: epoch

⏱️  Expected training time: ~10-12 hours (with oversampled data)
💡 Class imbalance fixes applied: weighted loss + oversampling + lower thresholds


---
## Step 4: Set Random Seed

In [18]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

def download_from_s3(bucket, key, local_path):
    """Download a file from S3 to local path."""
    print(f"Downloading s3://{bucket}/{key} to {local_path}...")
    s3.download_file(bucket, key, local_path)
    print(f" Downloaded {os.path.basename(local_path)}")
    return local_path

def download_s3_directory(bucket, prefix, local_dir):
    """Download all files from S3 prefix to local directory."""
    print(f"Downloading s3://{bucket}/{prefix}/ to {local_dir}/...")
    os.makedirs(local_dir, exist_ok=True)
    
    # List all objects with the prefix
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
    
    file_count = 0
    for page in pages:
        if 'Contents' not in page:
            continue
        for obj in page['Contents']:
            key = obj['Key']
            # Skip if it's a directory marker
            if key.endswith('/'):
                continue
            # Create local file path
            relative_path = os.path.relpath(key, prefix)
            local_file = os.path.join(local_dir, relative_path)
            os.makedirs(os.path.dirname(local_file), exist_ok=True)
            # Download
            s3.download_file(bucket, key, local_file)
            file_count += 1
    
    print(f"✅ Downloaded {file_count} files to {local_dir}")
    return local_dir
    
set_seed(SEED)
print(f"✅ Random seed set to {SEED}")
print(f"✅ S3 helper functions ready")

✅ Random seed set to 42
✅ S3 helper functions ready


---
## Step 5: Create Dataset Class

In [4]:
class SepsisDataset(Dataset):
    """Dataset for sepsis prediction with S3 support"""
    
    def __init__(self, data_path, tokenizer, max_length=2048, use_sampling=False, sample_ratio=1.0):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_sampling = use_sampling
        self.sample_ratio = sample_ratio
        self.data = self.load_data(data_path)
        
        # Apply sampling if requested
        if self.use_sampling and self.sample_ratio < 1.0:
            original_size = len(self.data)
            sample_size = int(original_size * self.sample_ratio)
            self.data = random.sample(self.data, sample_size)
            print(f"🔸 Sampled {sample_size}/{original_size} samples ({self.sample_ratio*100:.0f}%)")
        
    def load_data(self, data_path):
        """Load data from JSONL file (local or S3)"""
        data = []
        
        # If S3 path, download to local cache first
        if data_path.startswith('s3://'):
            # Parse S3 path
            s3_path = data_path.replace('s3://', '')
            bucket = s3_path.split('/')[0]
            key = '/'.join(s3_path.split('/')[1:])
            
            # Local cache path
            local_file = os.path.join(LOCAL_DATA_DIR, os.path.basename(key))
            
            # Download if not cached
            if not os.path.exists(local_file):
                download_from_s3(bucket, key, local_file)
            else:
                print(f"Using cached {os.path.basename(local_file)}")
            
            data_path = local_file
        
        # Load from local file
        with open(data_path, 'r') as f:
            for line in f:
                item = json.loads(line)
                # Handle both formats
                if 'output' in item:
                    text = item['input']
                    labels = [item['output']['h6'], item['output']['h12'], 
                             item['output']['h24'], item['output']['h48']]
                else:
                    text = item.get('text', item.get('input', ''))
                    labels = [item['h6'], item['h12'], item['h24'], item['h48']]
                
                data.append({
                    'text': text,
                    'labels': labels
                })
        
        print(f"✅ Loaded {len(data)} samples from {os.path.basename(data_path)}")
        return data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Tokenize
        encoding = self.tokenizer(
            item['text'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(item['labels'], dtype=torch.float32)
        }

print("✅ Dataset class created with S3 support!")

✅ Dataset class created with S3 support!


---
## Step 6: Load Model and Tokenizer

This loads Meditron-7B with 4-bit quantization (QLoRA).

In [5]:
print("Loading tokenizer...")

# Download model from S3 if needed
model_path = f"s3://{S3_BUCKET}/{S3_MODEL_PREFIX}"
local_model_path = LOCAL_CACHE_DIR

# Check if model exists locally, if not download from S3
model_files = ['config.json', 'tokenizer.json', 'tokenizer_config.json']
needs_download = not all(os.path.exists(os.path.join(local_model_path, f)) for f in model_files)

if needs_download:
    print(f"Downloading model from {model_path}...")
    download_s3_directory(S3_BUCKET, S3_MODEL_PREFIX, local_model_path)
else:
    print("Using cached model files")

# Load tokenizer from local cache
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("✅ Tokenizer loaded!")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}")

Loading tokenizer...
Using cached model files
✅ Tokenizer loaded!
Vocab size: 32017
Pad token: <PAD>


In [6]:
print("Loading model with 4-bit quantization...")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load base model from local cache (downloaded from S3)
base_model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for k-bit training
base_model = prepare_model_for_kbit_training(base_model)

print("✅ Base model loaded!")
print(f"Model dtype: {base_model.dtype}")

Loading model with 4-bit quantization...


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Base model loaded!
Model dtype: torch.float32


---
## Step 7: Add LoRA Adapters

In [7]:
# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

# Add LoRA adapters
base_model = get_peft_model(base_model, lora_config)

print("✅ LoRA adapters added!")
base_model.print_trainable_parameters()

✅ LoRA adapters added!
trainable params: 159,907,840 || all params: 6,898,462,720 || trainable%: 2.3180


---
## Step 8: Add Classification Head

In [8]:
class SepsisClassificationModel(nn.Module):
    """Model with classification head for sepsis prediction"""
    
    def __init__(self, base_model, num_labels=4):
        super().__init__()
        self.base_model = base_model
        self.num_labels = num_labels
        
        # Get hidden size
        self.hidden_size = base_model.config.hidden_size
        
        # Classification head
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        
        # Move classifier to same device as model
        self.classifier.to(base_model.device)
        
    def forward(self, input_ids, attention_mask):
        # Get base model outputs
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        # Get last hidden state
        hidden_states = outputs.hidden_states[-1]
        
        # Pool: take last token
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = hidden_states.shape[0]
        pooled_output = hidden_states[range(batch_size), sequence_lengths]
        
        # Match dtype and device
        self.classifier = self.classifier.to(
            device=pooled_output.device,
            dtype=pooled_output.dtype
        )
        
        # Classification
        logits = self.classifier(pooled_output)
        
        return {'logits': logits}

# Create model
model = SepsisClassificationModel(base_model, num_labels=4)
model.eval()  # Start in eval mode

print("✅ Classification model created!")
print(f"Hidden size: {model.hidden_size}")
print(f"Output labels: {model.num_labels}")

✅ Classification model created!
Hidden size: 4096
Output labels: 4


---
## Step 9: Load Data

In [9]:
import random
print("Loading datasets from S3...")

# Create S3 paths
train_s3_path = f"s3://{S3_BUCKET}/{S3_DATA_PREFIX}/{TRAIN_FILE}"
val_s3_path = f"s3://{S3_BUCKET}/{S3_DATA_PREFIX}/{VAL_FILE}"

print(f"Train: {train_s3_path}")
print(f"Val: {val_s3_path}")

# Create datasets (will auto-download from S3)
train_dataset = SepsisDataset(
    train_s3_path,
    tokenizer,
    max_length=MAX_LENGTH,
    use_sampling=USE_DATA_SAMPLING,
    sample_ratio=SAMPLE_RATIO
)

val_dataset = SepsisDataset(
    val_s3_path,
    tokenizer,
    max_length=MAX_LENGTH,
    use_sampling=False  # Don't sample validation data
)

print(f"\n✅ Datasets loaded!")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Loading datasets from S3...
Train: s3://mimic-data-nahiyan/data/sepsis_top_n_train.jsonl
Val: s3://mimic-data-nahiyan/data/sepsis_top_n_val.jsonl
Using cached sepsis_top_n_train.jsonl
✅ Loaded 4000 samples from sepsis_top_n_train.jsonl
Using cached sepsis_top_n_val.jsonl
✅ Loaded 500 samples from sepsis_top_n_val.jsonl

✅ Datasets loaded!
Train samples: 4000
Val samples: 500
Train batches: 2000
Val batches: 250


In [10]:
# ========== OVERSAMPLE POSITIVE CASES ==========

def oversample_positives(dataset, oversample_ratio=5.0):
    """Oversample positive cases to balance the dataset"""
    import random
    
    positive_indices = []
    negative_indices = []
    
    for idx in range(len(dataset)):
        labels = dataset.data[idx]['labels']
        # If any label is positive (sepsis detected at any time horizon)
        if any(labels):
            positive_indices.append(idx)
        else:
            negative_indices.append(idx)
    
    print(f"📊 Original distribution:")
    print(f"   Positive samples: {len(positive_indices)}")
    print(f"   Negative samples: {len(negative_indices)}")
    print(f"   Ratio: {len(negative_indices)/len(positive_indices):.1f}:1 (negative:positive)")
    
    # Oversample positives by repeating them
    num_positive_repeats = int(oversample_ratio)
    oversampled_indices = negative_indices + (positive_indices * num_positive_repeats)
    
    # Shuffle to mix positive and negative samples
    random.shuffle(oversampled_indices)
    
    # Create new dataset with oversampled indices
    oversampled_data = [dataset.data[idx] for idx in oversampled_indices]
    dataset.data = oversampled_data
    
    # Count new distribution
    new_positive = len(positive_indices) * num_positive_repeats
    new_negative = len(negative_indices)
    
    print(f"\n✅ After oversampling ({oversample_ratio}x):")
    print(f"   Positive samples: {new_positive}")
    print(f"   Negative samples: {new_negative}")
    print(f"   New ratio: {new_negative/new_positive:.1f}:1 (negative:positive)")
    print(f"   Total samples: {len(dataset)}")
    print(f"   This will help the model learn sepsis patterns!")
    
    return dataset

# Apply oversampling to training set only (NOT validation)
print("Applying oversampling to training set...")
train_dataset = oversample_positives(train_dataset, oversample_ratio=5.0)

# Recreate dataloader with oversampled data
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

print(f"\n📦 New train batches: {len(train_loader)}")
print("   (Note: Training will take longer due to more samples)")

Applying oversampling to training set...
📊 Original distribution:
   Positive samples: 187
   Negative samples: 3813
   Ratio: 20.4:1 (negative:positive)

✅ After oversampling (5.0x):
   Positive samples: 935
   Negative samples: 3813
   New ratio: 4.1:1 (negative:positive)
   Total samples: 4748
   This will help the model learn sepsis patterns!

📦 New train batches: 2374
   (Note: Training will take longer due to more samples)


In [11]:
# ========== CHECK CLASS DISTRIBUTION ==========

def check_class_distribution(dataset, name="Dataset"):
    """Print detailed class distribution for all time horizons"""
    class_counts = [0, 0, 0, 0]
    
    for item in dataset.data:
        labels = item['labels']
        for i, label in enumerate(labels):
            if label == 1:
                class_counts[i] += 1
    
    print(f"\n{'='*60}")
    print(f"{name} - Class Distribution")
    print(f"{'='*60}")
    print(f"{'Horizon':<10} {'Positive':<12} {'Negative':<12} {'% Positive':<12}")
    print("-" * 60)
    
    horizons = ['h6', 'h12', 'h24', 'h48']
    for i, horizon in enumerate(horizons):
        pos_count = class_counts[i]
        neg_count = len(dataset) - pos_count
        pos_pct = 100 * pos_count / len(dataset)
        print(f"{horizon:<10} {pos_count:<12} {neg_count:<12} {pos_pct:<12.2f}%")
    
    print("-" * 60)
    print(f"{'Total':<10} {len(dataset)} samples")
    print(f"{'='*60}\n")

# Check distribution for training and validation sets
check_class_distribution(train_dataset, "Training Set (After Oversampling)")
check_class_distribution(val_dataset, "Validation Set (Original)")

print("💡 Note: Training set has oversampled positive cases to help the model learn!")
print("💡 Validation set remains unchanged for fair evaluation.")


Training Set (After Oversampling) - Class Distribution
Horizon    Positive     Negative     % Positive  
------------------------------------------------------------
h6         190          4558         4.00        %
h12        370          4378         7.79        %
h24        615          4133         12.95       %
h48        935          3813         19.69       %
------------------------------------------------------------
Total      4748 samples


Validation Set (Original) - Class Distribution
Horizon    Positive     Negative     % Positive  
------------------------------------------------------------
h6         6            494          1.20        %
h12        9            491          1.80        %
h24        14           486          2.80        %
h48        21           479          4.20        %
------------------------------------------------------------
Total      500 samples

💡 Note: Training set has oversampled positive cases to help the model learn!
💡 Validation set r

---
## Step 10: Define Loss Function

In [12]:
class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance"""
    
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        
        loss = alpha_weight * focal_weight * bce_loss
        return loss.mean()

criterion = FocalLoss(alpha=0.25, gamma=2.0)
print("✅ Focal loss created!")

✅ Focal loss created!


---
## Step 11: Setup Optimizer and Scheduler

In [13]:
# Optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

# Scheduler
num_training_steps = len(train_loader) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * 0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_training_steps - num_warmup_steps,
    eta_min=1e-7
)

print("✅ Optimizer and scheduler created!")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

✅ Optimizer and scheduler created!
Learning rate: 5e-05
Training steps: 7122
Warmup steps: 356


---
## Step 12: Evaluation Function

In [14]:
def evaluate(model, dataloader, device):
    """Evaluate model on validation set"""
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].cpu().numpy()
            
            # Forward pass
            outputs = model(input_ids, attention_mask)
            logits = outputs['logits']
            
            # Get predictions
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_preds.append(preds)
            all_labels.append(labels)
    
    # Concatenate
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Compute metrics
    metrics = {}
    horizons = ['h6', 'h12', 'h24', 'h48']
    
    for i, horizon in enumerate(horizons):
        y_true = all_labels[:, i]
        y_pred = all_preds[:, i]
        
        metrics[f'{horizon}_accuracy'] = accuracy_score(y_true, y_pred)
        metrics[f'{horizon}_precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f'{horizon}_recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics[f'{horizon}_f1'] = f1_score(y_true, y_pred, zero_division=0)
    
    # Average metrics
    metrics['avg_accuracy'] = np.mean([metrics[f'{h}_accuracy'] for h in horizons])
    metrics['avg_precision'] = np.mean([metrics[f'{h}_precision'] for h in horizons])
    metrics['avg_recall'] = np.mean([metrics[f'{h}_recall'] for h in horizons])
    metrics['avg_f1'] = np.mean([metrics[f'{h}_f1'] for h in horizons])
    
    return metrics

print("✅ Evaluation function defined!")

✅ Evaluation function defined!


---
## Step 13: Training Loop

**This will take 5-7 hours for 3 epochs!**

In [15]:
# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Training state
global_step = 0
best_f1 = 0.0
train_losses = []

print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Evaluation strategy: {EVALUATION_STRATEGY} (once per epoch)")
print("="*70 + "\n")

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")
    
    # Training
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(input_ids, attention_mask)
        logits = outputs['logits']
        
        # Compute loss
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.3)
        
        # Update weights every gradient_accumulation_steps
        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
        
        # Track loss
        epoch_loss += loss.item()
        num_batches += 1
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'avg_loss': f"{epoch_loss/num_batches:.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
    
    # End of epoch evaluation
    print(f"\n{'='*70}")
    print(f"END OF EPOCH {epoch+1} EVALUATION")
    print(f"{'='*70}")
    
    avg_epoch_loss = epoch_loss / num_batches
    print(f"Average epoch loss: {avg_epoch_loss:.4f}")
    
    # Evaluate
    torch.cuda.empty_cache()
    metrics = evaluate(model, val_loader, device)
    
    # Print metrics
    print("\nValidation Metrics:")
    print(f"  h6  - F1: {metrics['h6_f1']:.4f}, Acc: {metrics['h6_accuracy']:.4f}")
    print(f"  h12 - F1: {metrics['h12_f1']:.4f}, Acc: {metrics['h12_accuracy']:.4f}")
    print(f"  h24 - F1: {metrics['h24_f1']:.4f}, Acc: {metrics['h24_accuracy']:.4f}")
    print(f"  h48 - F1: {metrics['h48_f1']:.4f}, Acc: {metrics['h48_accuracy']:.4f}")
    print(f"\n  AVG - F1: {metrics['avg_f1']:.4f}, Acc: {metrics['avg_accuracy']:.4f}")
    
    # Save best model
    if metrics['avg_f1'] > best_f1:
        best_f1 = metrics['avg_f1']
        print(f"\n✅ New best model! Avg F1: {best_f1:.4f}")
        
        # Save model
        save_path = Path(OUTPUT_DIR) / "best_model"
        save_path.mkdir(parents=True, exist_ok=True)
        model.base_model.save_pretrained(str(save_path))
        tokenizer.save_pretrained(str(save_path))
        print(f"Model saved to {save_path}")
    
    # Save checkpoint
    checkpoint_path = Path(OUTPUT_DIR) / f"checkpoint_epoch_{epoch+1}"
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    model.base_model.save_pretrained(str(checkpoint_path))
    tokenizer.save_pretrained(str(checkpoint_path))
    print(f"\nCheckpoint saved to {checkpoint_path}")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Best Avg F1: {best_f1:.4f}")
print(f"Models saved to: {OUTPUT_DIR}")
print("="*70)

Training on: cuda

STARTING TRAINING
Epochs: 3
Batch size: 2
Gradient accumulation: 32
Effective batch size: 64
Evaluation strategy: epoch (once per epoch)


EPOCH 1/3


Epoch 1:   0%|          | 0/2374 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/ubuntu/nahiyan_workspace/sepsis_prediction/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



END OF EPOCH 1 EVALUATION
Average epoch loss: 0.0682


Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]


Validation Metrics:
  h6  - F1: 0.0000, Acc: 0.9880
  h12 - F1: 0.0000, Acc: 0.9820
  h24 - F1: 0.0000, Acc: 0.9700
  h48 - F1: 0.0000, Acc: 0.9580

  AVG - F1: 0.0000, Acc: 0.9745

Checkpoint saved to /home/ubuntu/nahiyan_workspace/sepsis_prediction/outputs/qlora_s3/checkpoint_epoch_1

EPOCH 2/3


... (6 outputs left out)

/home/ubuntu/nahiyan_workspace/sepsis_prediction/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



END OF EPOCH 3 EVALUATION
Average epoch loss: 0.0327


Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]


Validation Metrics:
  h6  - F1: 0.0000, Acc: 0.9880
  h12 - F1: 0.0000, Acc: 0.9820
  h24 - F1: 0.0000, Acc: 0.9700
  h48 - F1: 0.0000, Acc: 0.9320

  AVG - F1: 0.0000, Acc: 0.9680

Checkpoint saved to /home/ubuntu/nahiyan_workspace/sepsis_prediction/outputs/qlora_s3/checkpoint_epoch_3

TRAINING COMPLETE!
Best Avg F1: 0.0132
Models saved to: /home/ubuntu/nahiyan_workspace/sepsis_prediction/outputs/qlora_s3


---
## Step 14: Final Evaluation

Load the best model and evaluate on test set (optional).

In [16]:
# Load test data from S3 (if available)
try:
    # Create S3 path for test data
    test_s3_path = f"s3://{S3_BUCKET}/{S3_DATA_PREFIX}/{TEST_FILE}"
    
    print(f"Loading test data from: {test_s3_path}")
    
    test_dataset = SepsisDataset(
        test_s3_path,
        tokenizer,
        max_length=MAX_LENGTH
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )
    
    print(f"Test samples: {len(test_dataset)}")
    
    # Evaluate
    print("\n" + "="*70)
    print("FINAL TEST SET EVALUATION")
    print("="*70)
    
    test_metrics = evaluate(model, test_loader, device)
    
    # Print metrics
    print("\nTest Set Metrics:")
    print(f"  h6  - F1: {test_metrics['h6_f1']:.4f}, Acc: {test_metrics['h6_accuracy']:.4f}")
    print(f"  h12 - F1: {test_metrics['h12_f1']:.4f}, Acc: {test_metrics['h12_accuracy']:.4f}")
    print(f"  h24 - F1: {test_metrics['h24_f1']:.4f}, Acc: {test_metrics['h24_accuracy']:.4f}")
    print(f"  h48 - F1: {test_metrics['h48_f1']:.4f}, Acc: {test_metrics['h48_accuracy']:.4f}")
    print(f"\n  AVG - F1: {test_metrics['avg_f1']:.4f}, Acc: {test_metrics['avg_accuracy']:.4f}")
    print("="*70)
    
except FileNotFoundError:
    print("❌ Test file not found in S3. Skipping final evaluation.")
except Exception as e:
    print(f"❌ Error loading test data: {e}")
    print("Skipping final evaluation.")

Loading test data from: s3://mimic-data-nahiyan/data/sepsis_top_n_test.jsonl
Using cached sepsis_top_n_test.jsonl
✅ Loaded 500 samples from sepsis_top_n_test.jsonl
Test samples: 500

FINAL TEST SET EVALUATION


Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]


Test Set Metrics:
  h6  - F1: 0.0000, Acc: 0.9820
  h12 - F1: 0.0000, Acc: 0.9720
  h24 - F1: 0.0000, Acc: 0.9620
  h48 - F1: 0.0976, Acc: 0.9260

  AVG - F1: 0.0244, Acc: 0.9605


---
## Step 15: Save Final Model

In [17]:
# Save final model
final_path = Path(OUTPUT_DIR) / "final_model"
final_path.mkdir(parents=True, exist_ok=True)

model.base_model.save_pretrained(str(final_path))
tokenizer.save_pretrained(str(final_path))

print(f"✅ Final model saved to: {final_path}")

✅ Final model saved to: /home/ubuntu/nahiyan_workspace/sepsis_prediction/outputs/qlora_s3/final_model


---
## Summary

### What you trained:
- ✅ Meditron-7B with QLoRA (4-bit quantization)
- ✅ Multi-label classification for sepsis prediction (h6, h12, h24, h48)
- ✅ Focal loss for class imbalance
- ✅ 3 epochs of training

### Models saved:
- `best_model/` - Best model based on validation F1
- `checkpoint_epoch_X/` - Checkpoints after each epoch
- `final_model/` - Final model after all training

### Next steps:
1. Use the model for inference on new patients
2. Compare with zero-shot/few-shot results
3. Deploy for clinical use

### To load and use the model:
```python
from peft import AutoPeftModelForCausalLM

# Load model
model = AutoPeftModelForCausalLM.from_pretrained(
    "outputs/qlora_notebook/best_model",
    device_map="auto"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "outputs/qlora_notebook/best_model"
)
```